In [3]:
pip install --user torchvision

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.3 MB 11.5 MB/s eta 0:00:01
   ------------------------------- -------- 3.4/4.3 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 9.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
!{sys.executable} -m pip install torchvision

Defaulting to user installation because normal site-packages is not writeable


In [6]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg' 
plt.style.use('seaborn-v0_8')

In [7]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [8]:
ratings_data = pd.read_csv('./ml-latest-small/ratings.csv')
movie_names_data = pd.read_csv('./ml-latest-small/movies.csv')

In [9]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [10]:
ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [11]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [12]:
user_enc.classes_[4], movie_enc.classes_[8871]

(np.int64(5), 'Toy Story (1995)')

In [13]:
for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(8871)] 4.0
[np.int64(0), np.int64(3661)] 4.0
[np.int64(0), np.int64(3845)] 4.0
[np.int64(0), np.int64(7523)] 5.0
[np.int64(0), np.int64(9119)] 5.0
[np.int64(0), np.int64(3252)] 3.0
[np.int64(0), np.int64(1284)] 5.0
[np.int64(0), np.int64(1337)] 4.0
[np.int64(0), np.int64(7180)] 5.0
[np.int64(0), np.int64(1535)] 5.0


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [15]:
num_users = len(X)
num_movies = len(X)

In [17]:
import numpy as np
import tensorflow as tf
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

# ✅ Define number of users and movies (IMPORTANT)
n_user = 1000     # example: total users
n_movies = 500    # example: total movies/items

def build_model():
    # Movie input
    movie_input = Input(shape=(1,), name="Movie-Input")
    movie_embedding = Embedding(input_dim=n_movies+1, output_dim=15, name="Movie-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Movies")(movie_embedding)

    # User input
    user_input = Input(shape=(1,), name="User-Input")
    user_embedding = Embedding(input_dim=n_user+1, output_dim=15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)

    # Dot product
    prod = Dot(axes=1, name="Dot-Product")([user_vec, movie_vec])

    # Dense layers
    prod = Dense(32, activation='relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16, activation='relu')(prod)
    prod = Dropout(0.5)(prod)

    output = Dense(1)(prod)

    # Model
    model = Model(inputs=[user_input, movie_input], outputs=output)
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    return model

# ✅ Build model
model = build_model()

# ✅ Summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ User-Input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie-Input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ User-Embedding      │ (None, 1, 15)     │     15,015 │ User-Input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie-Embedding     │ (None, 1, 15)     │      7,515 │ Movie-Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten-Users       │ (None, 15)        │          0 │ User-Embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten-Movies      │ (None, 15)        │          0 │ Movie-Embedding[… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dot-Product (Dot)   │ (None, 1)         │          0 │ Flatten-Users[0]… │
│                     │                   │            │ Flatten-Movies[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │         64 │ Dot-Product[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 16)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         17 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,139 (90.39 KB)

 Trainable params: 23,139 (90.39 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Dropout
from keras.models import Model

# -------------------------------
# 1. Example Dataset (Replace with your dataset)
# -------------------------------
# Format: [user_id, movie_id]
data = np.array([
    [101, 5001],
    [102, 5002],
    [103, 5003],
    [101, 5002],
    [102, 5003],
    [103, 5001],
])

ratings = np.array([5, 4, 3, 4, 5, 2])  # target values

# -------------------------------
# 2. Encode IDs (VERY IMPORTANT)
# -------------------------------
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

users = user_encoder.fit_transform(data[:, 0])
movies = movie_encoder.fit_transform(data[:, 1])

# Combine back
X = np.column_stack((users, movies))
Y = ratings

# -------------------------------
# 3. Train-Test Split
# -------------------------------
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# -------------------------------
# 4. Get counts for Embedding
# -------------------------------
n_user = len(user_encoder.classes_)
n_movies = len(movie_encoder.classes_)

# -------------------------------
# 5. Build Model
# -------------------------------
def build_model():
    user_input = Input(shape=(1,), name="User-Input")
    user_embedding = Embedding(n_user, 15, name="User-Embedding")(user_input)
    user_vec = Flatten()(user_embedding)

    movie_input = Input(shape=(1,), name="Movie-Input")
    movie_embedding = Embedding(n_movies, 15, name="Movie-Embedding")(movie_input)
    movie_vec = Flatten()(movie_embedding)

    # Dot product
    prod = Dot(axes=1)([user_vec, movie_vec])

    # Dense layers
    prod = Dense(32, activation='relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16, activation='relu')(prod)
    prod = Dropout(0.5)(prod)

    output = Dense(1)(prod)

    model = Model(inputs=[user_input, movie_input], outputs=output)
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    return model

model = build_model()
model.summary()

# -------------------------------
# 6. Checkpoint Callback (FIXED)
# -------------------------------
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

# -------------------------------
# 7. Train Model
# -------------------------------
history = model.fit(
    [X_train[:, 0], X_train[:, 1]],
    Y_train,
    epochs=10,
    batch_size=2,
    validation_data=([X_test[:, 0], X_test[:, 1]], Y_test),
    callbacks=[checkpoint]
)

# -------------------------------
# 8. Prediction Example
# -------------------------------
# Predict for user 101 & movie 5001
user_id = 101
movie_id = 5001

user_encoded = user_encoder.transform([user_id])
movie_encoded = movie_encoder.transform([movie_id])

prediction = model.predict([user_encoded, movie_encoded])
print("Predicted Rating:", prediction[0][0])

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ User-Input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie-Input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ User-Embedding      │ (None, 1, 15)     │         45 │ User-Input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie-Embedding     │ (None, 1, 15)     │         45 │ Movie-Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 15)        │          0 │ User-Embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 15)        │          0 │ Movie-Embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │         64 │ dot[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │        528 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 16)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         17 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 699 (2.73 KB)

 Trainable params: 699 (2.73 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/2 ━━━━━━━━━━━━━━━━━━━━ 1s 2s/step - loss: 9.9992 - mae: 2.9998
Epoch 1: val_loss improved from None to 20.46087, saving model to ./checkpoint.weights.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 454ms/step - loss: 13.4950 - mae: 3.4993 - val_loss: 20.4609 - val_mae: 4.4956
Epoch 2/10
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 6.4820 - mae: 2.4966
Epoch 2: val_loss improved from 20.46087 to 20.42215, saving model to ./checkpoint.weights.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 13.4632 - mae: 3.4951 - val_loss: 20.4222 - val_mae: 4.4913
Epoch 3/10
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 20.4474 - mae: 4.4940
Epoch 3: val_loss improved from 20.42215 to 20.37827, saving model to ./checkpoint.weights.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 13.4466 - mae: 3.4917 - val_loss: 20.3783 - val_mae: 4.4865
Epoch 4/10
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 16.8603 - mae: 3.9826
Epoch 4: val_loss improved from 20.37827 to 20.33065, saving model to ./checkpoint.weight

In [21]:
X_test[:5], Y_test[:5]

(array([[0, 0],
        [1, 1]]),
 array([5, 4]))

In [22]:
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step


In [24]:
# Make predictions (example: first 5 samples)
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

# Print predictions vs actual values
print("Predictions:\n", predictions.flatten())
print("\nActual:\n", Y_test[:5])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predictions:
 [0.05095888 0.0507005 ]

Actual:
 [5 4]


In [25]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [27]:
import numpy as np

def extract_true_ratings(user_id, X, Y):
    # Find indices where user_id matches
    indices = np.where(X[:, 0] == user_id)[0]
    
    # Get corresponding movie IDs and ratings
    user_movies = X[indices][:, 1]
    user_ratings = Y[indices]
    
    return user_movies, user_ratings

In [28]:
def extract_true_ratings(user_id, X_test):
    
    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [29]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [30]:
test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]
# userid_rating_data

In [31]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

In [32]:
dictionary = {"user_id": [test_user_id]*len(movie_ids),
              "movie_id": movie_ids,
              "movie_name":movie_names,
              "predicted_ratings":predictions,
              "true_ratings": extract_true_ratings(test_user_id, X_test) 
              }

In [33]:
prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)

,user_id,movie_id,movie_name,predicted_ratings,true_ratings
